# 🏗️ Data Mesh Demo — Overview & Infrastructure Setup

## What is Data Mesh?
Data Mesh is a **decentralized** approach to data architecture where **domain teams own their data as products**. Instead of a central data team bottleneck, each domain is responsible for producing high-quality, discoverable data products within **their own catalog**.

## This Demo
We simulate **two data products** from two independent domains, plus a **federated hub**:

| Catalog | Domain | Source System | Data Product | Owner |
|---------|--------|--------------|--------------|-------|
| `adoit_product` | Enterprise Architecture | **ADOIT** | Application Landscape | EA Team |
| `snow_product` | IT Service Management | **SNOW** (ServiceNow) | Service Health | ITSM Team |
| `data_mesh_hub` | Cross-Domain | Both | Application Risk Profile | Platform Team |

## Architecture — Separate Catalogs per Domain
```
┌─────────────────────────┐  ┌─────────────────────────┐  ┌─────────────────────────┐
│  adoit_product (Catalog) │  │  snow_product (Catalog)  │  │  data_mesh_hub (Catalog) │
│  Owner: EA Team          │  │  Owner: ITSM Team        │  │  Owner: Platform Team     │
├─────────────────────────┤  ├─────────────────────────┤  ├─────────────────────────┤
│ bronze                  │  │ bronze                  │  │ cross_domain            │
│  └ raw_applications     │  │  └ raw_incidents         │  │  └ application_risk_    │
│  └ raw_capabilities     │  │  └ raw_change_requests  │  │      profile             │
│  └ raw_app_cap_map      │  ├─────────────────────────┤  ├─────────────────────────┤
├─────────────────────────┤  │ silver                  │  │ platform                │
│ silver                  │  │  └ incidents             │  │  └ data_quality_log     │
│  └ applications         │  │  └ change_requests      │  └─────────────────────────┘
│  └ business_capabilities│  ├─────────────────────────┤
├─────────────────────────┤  │ gold                    │
│ gold                    │  │  └ service_health ─────┤
│  └ application_      ───┤  └─────────────────────────┘
│      landscape           │         │                     │
└─────────────────────────┘         │                     │
            │                          │                     │
            └────────────────────────┴─────────────────────┘
              Cross-catalog JOINs in data_mesh_hub
```

**Why separate catalogs?** Each domain team has **full autonomy** over their catalog — they manage their own schemas, tables, quality, and access controls. Cross-domain consumption happens via **cross-catalog JOINs** in Unity Catalog, which work seamlessly.

## Data Product Characteristics Mapped to Databricks
| Characteristic | How We Demonstrate It |
|---|---|
| **Clear Owner** | Catalog-level ownership, TBLPROPERTIES (`data_product.owner`), UC Tags |
| **Business Purpose** | Table COMMENT, TBLPROPERTIES (`data_product.business_purpose`) |
| **Defined Consumers** | TBLPROPERTIES (`data_product.consumers`) |
| **Documented Definitions** | Column COMMENTs on every column, table COMMENTs |
| **Quality Checks** | Quality score columns, validation queries, `data_quality_log` table |
| **Access Controls** | GRANT per catalog — domain team owns catalog, consumers get SELECT on gold |
| **Lineage** | Medallion (Bronze→Silver→Gold), cross-catalog lineage via UC |
| **Discoverability** | UC Tags at catalog/schema/table level, Genie Space, Dashboard |
| **Reuse Across Teams** | `data_mesh_hub` joins gold products from both domain catalogs |
| **Reliability / Freshness** | SLA in TBLPROPERTIES, freshness checks in quality log |

## Step 1: Create Domain Catalogs & Schemas
Each domain owns **its own catalog** — true domain autonomy. Schemas follow **Medallion Architecture** (Bronze → Silver → Gold).

In [0]:
%sql
-- ============================================================
-- DOMAIN CATALOG: adoit_product (EA Team owns this entirely)
-- ============================================================
CREATE CATALOG IF NOT EXISTS adoit_product 
MANAGED LOCATION 'abfss://data-products@sadpsddbxdev.dfs.core.windows.net/adoit_product'
COMMENT 'ADOIT Data Product Catalog | Owner: Enterprise Architecture Team | Domain: Enterprise Architecture';

CREATE SCHEMA IF NOT EXISTS adoit_product.bronze COMMENT 'Raw Zone | Ingested from ADOIT EA tool | Owner: EA Team';
CREATE SCHEMA IF NOT EXISTS adoit_product.silver COMMENT 'Curated Zone | Cleansed and enriched | Owner: EA Team';
CREATE SCHEMA IF NOT EXISTS adoit_product.gold   COMMENT 'Product Zone | Business-ready data products | Owner: EA Team';

-- ============================================================
-- DOMAIN CATALOG: snow_product (ITSM Team owns this entirely)
-- ============================================================
CREATE CATALOG IF NOT EXISTS snow_product 
MANAGED LOCATION 'abfss://data-products@sadpsddbxdev.dfs.core.windows.net/snow_product'
COMMENT 'SNOW Data Product Catalog | Owner: IT Service Management Team | Domain: ITSM';

CREATE SCHEMA IF NOT EXISTS snow_product.bronze COMMENT 'Raw Zone | Ingested from ServiceNow | Owner: ITSM Team';
CREATE SCHEMA IF NOT EXISTS snow_product.silver COMMENT 'Curated Zone | Cleansed and enriched | Owner: ITSM Team';
CREATE SCHEMA IF NOT EXISTS snow_product.gold   COMMENT 'Product Zone | Business-ready data products | Owner: ITSM Team';

-- ============================================================
-- HUB CATALOG: data_mesh_hub (Platform Team — cross-domain)
-- ============================================================
CREATE CATALOG IF NOT EXISTS data_mesh_hub 
MANAGED LOCATION 'abfss://data-products@sadpsddbxdev.dfs.core.windows.net/data_mesh_hub'
COMMENT 'Data Mesh Hub | Owner: Data Platform Team | Cross-domain federated products and platform services';

CREATE SCHEMA IF NOT EXISTS data_mesh_hub.cross_domain COMMENT 'Cross-Domain Products | Federated products combining domain catalogs | Owner: Data Platform Team';
CREATE SCHEMA IF NOT EXISTS data_mesh_hub.platform     COMMENT 'Platform Services | Quality monitoring, product registry | Owner: Data Platform Team';

## Step 2: Tag Catalogs, Schemas & Tables for Discoverability
Unity Catalog **Tags** at every level — catalog, schema, and table — make assets discoverable via search and Catalog Explorer.

In [0]:
%sql
-- Tag CATALOGS with domain ownership and mesh role
ALTER CATALOG adoit_product SET TAGS ('domain' = 'Enterprise Architecture', 'owner' = 'EA Team', 'data_mesh_role' = 'Domain Catalog');
ALTER CATALOG snow_product  SET TAGS ('domain' = 'IT Service Management', 'owner' = 'ITSM Team', 'data_mesh_role' = 'Domain Catalog');
ALTER CATALOG data_mesh_hub SET TAGS ('domain' = 'Cross-Domain', 'owner' = 'Data Platform Team', 'data_mesh_role' = 'Hub Catalog');

-- Tag SCHEMAS with domain, owner, and tier
ALTER SCHEMA adoit_product.bronze SET TAGS ('domain' = 'Enterprise Architecture', 'owner' = 'EA Team', 'tier' = 'bronze');
ALTER SCHEMA adoit_product.silver SET TAGS ('domain' = 'Enterprise Architecture', 'owner' = 'EA Team', 'tier' = 'silver');
ALTER SCHEMA adoit_product.gold   SET TAGS ('domain' = 'Enterprise Architecture', 'owner' = 'EA Team', 'tier' = 'gold');
ALTER SCHEMA snow_product.bronze  SET TAGS ('domain' = 'IT Service Management', 'owner' = 'ITSM Team', 'tier' = 'bronze');
ALTER SCHEMA snow_product.silver  SET TAGS ('domain' = 'IT Service Management', 'owner' = 'ITSM Team', 'tier' = 'silver');
ALTER SCHEMA snow_product.gold    SET TAGS ('domain' = 'IT Service Management', 'owner' = 'ITSM Team', 'tier' = 'gold');
ALTER SCHEMA data_mesh_hub.cross_domain SET TAGS ('domain' = 'Cross-Domain', 'owner' = 'Data Platform Team', 'tier' = 'gold');
ALTER SCHEMA data_mesh_hub.platform     SET TAGS ('domain' = 'Platform', 'owner' = 'Data Platform Team', 'tier' = 'platform');

-- NOTE: Gold table tagging is done in 08_Advanced_Governance (after tables are created by pipeline notebooks)

## Step 3: Access Control Patterns
With **separate catalogs**, each domain team has **full autonomy** over their access controls:

```sql
-- ============================================================
-- EA TEAM: Full control of their adoit_product catalog
-- ============================================================
GRANT ALL PRIVILEGES ON CATALOG adoit_product TO `ea-team`;

-- Consumers get READ-ONLY on the gold data products
GRANT USE CATALOG  ON CATALOG adoit_product TO `cto-office`;
GRANT USE SCHEMA   ON SCHEMA  adoit_product.gold TO `cto-office`;
GRANT SELECT       ON TABLE   adoit_product.gold.application_landscape TO `cto-office`;

-- ============================================================
-- ITSM TEAM: Full control of their snow_product catalog
-- ============================================================
GRANT ALL PRIVILEGES ON CATALOG snow_product TO `itsm-team`;

-- Consumers get READ-ONLY on gold
GRANT USE CATALOG  ON CATALOG snow_product TO `risk-team`;
GRANT USE SCHEMA   ON SCHEMA  snow_product.gold TO `risk-team`;
GRANT SELECT       ON TABLE   snow_product.gold.service_health TO `risk-team`;

-- ============================================================
-- PLATFORM TEAM: Manages the hub, grants read to everyone
-- ============================================================
GRANT ALL PRIVILEGES ON CATALOG data_mesh_hub TO `data-platform-team`;

GRANT USE CATALOG ON CATALOG data_mesh_hub TO `cto-office`;
GRANT USE CATALOG ON CATALOG data_mesh_hub TO `risk-team`;
GRANT SELECT ON TABLE data_mesh_hub.cross_domain.application_risk_profile TO `cto-office`;
GRANT SELECT ON TABLE data_mesh_hub.cross_domain.application_risk_profile TO `risk-team`;
```

**Key difference from single-catalog**: The EA Team can manage `adoit_product` access without involving the Platform Team. True domain autonomy.

## Step 4: Inspect the Data Product Metadata
Data product properties are embedded in Unity Catalog, making them discoverable via `DESCRIBE EXTENDED` or the Catalog Explorer UI.

In [0]:
%sql
-- Verify catalogs and schemas created successfully
SHOW SCHEMAS IN adoit_product;

In [0]:
%sql
-- Verify SNOW schemas
SHOW SCHEMAS IN snow_product;

In [0]:
%sql
-- Verify Hub schemas
SHOW SCHEMAS IN data_mesh_hub;

## Next Steps
Run the domain-specific notebooks to see the full pipeline in action:
- **01_ADOIT_Data_Product_Pipeline** — Bronze→Silver→Gold within `adoit_product` catalog
- **02_SNOW_Data_Product_Pipeline** — Bronze→Silver→Gold within `snow_product` catalog
- **03_Cross_Domain_Data_Product** — Cross-catalog federated consumption in `data_mesh_hub`
- **04_Data_Quality_Monitoring** — Quality checks across all 3 catalogs